In [8]:
# =============================================================
# Fase 3 — Segmentación y clasificación de cobertura de manglar
# Pipeline multilenguaje CGSM (2018–2025)
# Lina María Quintero Fonseca | Maestría en Geomática, UNAL
# =============================================================

import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import os

ee.Initialize()

# --- AOI ---
aoi = ee.Geometry.Polygon([[
    [-74.8700, 11.0600], [-74.8200, 11.0700], [-74.7500, 11.0650],
    [-74.6800, 11.0600], [-74.6000, 11.0500], [-74.5300, 11.0400],
    [-74.4800, 11.0350], [-74.4300, 11.0300], [-74.3800, 11.0200],
    [-74.3200, 11.0100], [-74.2500, 10.9900], [-74.2000, 10.9500],
    [-74.1700, 10.9000], [-74.1500, 10.8500], [-74.1300, 10.8000],
    [-74.1200, 10.7500], [-74.1100, 10.7000], [-74.1000, 10.6500],
    [-74.1200, 10.6000], [-74.1500, 10.5500], [-74.2000, 10.5000],
    [-74.2500, 10.4500], [-74.3000, 10.4000], [-74.3500, 10.3500],
    [-74.4500, 10.3400], [-74.5500, 10.3800], [-74.6200, 10.4200],
    [-74.7000, 10.4800], [-74.7500, 10.5500], [-74.8000, 10.6500],
    [-74.8500, 10.7500], [-74.8700, 10.8500], [-74.8800, 10.9500],
    [-74.8700, 11.0600]
]])

# --- Estaciones INVEMAR ---
subzonas = {
    'Isla_Boqueron': ee.Geometry.Point([-74.298, 10.962]),
    'Punta_Cerro': ee.Geometry.Point([-74.283, 10.973]),
    'Punta_Chino': ee.Geometry.Point([-74.305, 10.912]),
    'Rio_Sevilla': ee.Geometry.Point([-74.325, 10.880]),
    'Cano_Palos': ee.Geometry.Point([-74.471, 10.758]),
    'CP_Pajarales': ee.Geometry.Point([-74.75, 10.85]),
    'Cano_Clarin': ee.Geometry.Point([-74.55, 10.55]),
    'VIPIS': ee.Geometry.Point([-74.65, 11.02]),
}

print("Fase 3 inicializada")
print(f"Estaciones: {len(subzonas)}")

Fase 3 inicializada
Estaciones: 8


In [4]:
# --- Colección Sentinel-2 con índices ---
def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    cmri = ndvi.subtract(ndwi).rename('CMRI')
    return image.addBands([ndvi, ndwi, cmri])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2_clouds)
      .map(add_indices))

# --- Tres composites por período ---
composite_degradacion = s2.filterDate('2020-07-01', '2020-12-31').median().clip(aoi)
composite_recuperacion = s2.filterDate('2022-01-01', '2022-06-30').median().clip(aoi)
composite_actual = s2.filterDate('2024-07-01', '2025-06-30').median().clip(aoi)

print("Composites definidos:")
print(f"  Degradación: 2020-S2 ({s2.filterDate('2020-07-01', '2020-12-31').size().getInfo()} imgs)")
print(f"  Recuperación: 2022-S1 ({s2.filterDate('2022-01-01', '2022-06-30').size().getInfo()} imgs)")
print(f"  Actual: 2024-2025 ({s2.filterDate('2024-07-01', '2025-06-30').size().getInfo()} imgs)")

Composites definidos:
  Degradación: 2020-S2 (32 imgs)
  Recuperación: 2022-S1 (54 imgs)
  Actual: 2024-2025 (85 imgs)


In [10]:
# --- Cargar períodos desde Fase 2 ---
periodos_df = pd.read_csv('../outputs/tables/periodos_criticos.csv')
periodos = {}
for _, row in periodos_df.iterrows():
    periodos[row['periodo']] = {
        'fecha_inicio': row['fecha_inicio'],
        'fecha_fin': row['fecha_fin']
    }
print("Períodos cargados desde Fase 2:")
for k, v in periodos.items():
    print(f"  {k}: {v['fecha_inicio']} a {v['fecha_fin']}")

Períodos cargados desde Fase 2:
  degradacion: 2020-07-01 a 2020-12-31
  recuperacion: 2022-01-01 a 2022-06-30
  actual: 2024-07-01 a 2025-06-30


In [11]:
# --- Mapa de NDVI comparativo: Degradación vs Actual ---
ndvi_deg = (s2.filterDate('2020-07-01', '2020-12-31')
            .select('NDVI').median().clip(aoi))
ndvi_act = (s2.filterDate('2024-07-01', '2025-06-30')
            .select('NDVI').median().clip(aoi))

ndvi_change = ndvi_act.subtract(ndvi_deg).rename('NDVI_change')

palette_ndvi = ['red', 'orange', 'yellow', 'lightgreen', 'green', 'darkgreen']
palette_change = ['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#91cf60', '#1a9850']

Map5 = geemap.Map(center=[10.75, -74.45], zoom=11)
Map5.addLayer(ndvi_deg, {'min': -0.2, 'max': 0.8, 'palette': palette_ndvi}, 'NDVI Degradación (2020-S2)')
Map5.addLayer(ndvi_act, {'min': -0.2, 'max': 0.8, 'palette': palette_ndvi}, 'NDVI Actual (2024-2025)')
Map5.addLayer(ndvi_change, {'min': -0.4, 'max': 0.4, 'palette': palette_change}, 'Cambio NDVI (Actual - Degradación)')

for name, geom in subzonas.items():
    Map5.addLayer(ee.Feature(geom.buffer(300)), {'color': 'white'}, name)

Map5

Map(center=[10.75, -74.45], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [12]:
# --- Clasificación de manglar por umbral NDVI/CMRI ---
def classify_mangrove(start_date, end_date, periodo_name):
    composite = s2.filterDate(start_date, end_date).median().clip(aoi)
    
    ndvi = composite.normalizedDifference(['B8', 'B4'])
    ndwi = composite.normalizedDifference(['B3', 'B8'])
    cmri = ndvi.subtract(ndwi)
    
    manglar = cmri.gt(0.3).And(ndvi.gt(0.4)).selfMask().rename('manglar')
    
    area = manglar.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=10,
        maxPixels=1e13
    ).get('manglar').getInfo()
    
    area_ha = area / 10000
    print(f"  {periodo_name}: {area_ha:,.0f} ha de manglar")
    return manglar

print("=== Clasificación de manglar por período ===\n")
manglar_deg = classify_mangrove('2020-07-01', '2020-12-31', 'Degradación (2020-S2)')
manglar_rec = classify_mangrove('2022-01-01', '2022-06-30', 'Recuperación (2022-S1)')
manglar_act = classify_mangrove('2024-07-01', '2025-06-30', 'Actual (2024-2025)')

Map6 = geemap.Map(center=[10.75, -74.45], zoom=11)
Map6.addLayer(manglar_deg, {'palette': ['#d32f2f']}, 'Manglar Degradación', opacity=0.7)
Map6.addLayer(manglar_rec, {'palette': ['#ff9800']}, 'Manglar Recuperación', opacity=0.7)
Map6.addLayer(manglar_act, {'palette': ['#1b5e20']}, 'Manglar Actual', opacity=0.7)
Map6

=== Clasificación de manglar por período ===

  Degradación (2020-S2): 368,124 ha de manglar
  Recuperación (2022-S1): 318,653 ha de manglar
  Actual (2024-2025): 377,200 ha de manglar


Map(center=[10.75, -74.45], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [31]:
# --- Exportar clasificación de manglar a Google Drive ---
for name, img, periodo in [
    ('manglar_degradacion', manglar_deg, '2020_S2'),
    ('manglar_recuperacion', manglar_rec, '2022_S1'),
    ('manglar_actual', manglar_act, '2024_2025')
]:
    task = ee.batch.Export.image.toDrive(
        image=img.unmask(0).toByte(),
        description=f'CGSM_{name}',
        folder='CGSM_data',
        region=aoi,
        scale=10,
        crs='EPSG:4326',
        maxPixels=1e13
    )
    task.start()
    print(f"Exportando {name}")

print("\n3 clasificaciones lanzadas a Google Drive")

Exportando manglar_degradacion
Exportando manglar_recuperacion
Exportando manglar_actual

3 clasificaciones lanzadas a Google Drive


In [32]:
# --- Verificar estado de tareas ---
tasks = ee.batch.Task.list()
ready = [t for t in tasks if t.state == 'READY']
running = [t for t in tasks if t.state == 'RUNNING']
completed = [t for t in tasks if t.state == 'COMPLETED']
failed = [t for t in tasks if t.state == 'FAILED']
print(f"READY: {len(ready)} | RUNNING: {len(running)} | COMPLETED: {len(completed)} | FAILED: {len(failed)}")
if failed:
    print("\nTareas fallidas:")
    for t in failed:
        print(f"  {t.config.get('description', 'N/A')}")

READY: 10 | RUNNING: 2 | COMPLETED: 53 | FAILED: 2

Tareas fallidas:
  CGSM_indices_2018_Q3
  CGSM_indices_2018_Q2


In [33]:
# --- Verificar RGBs disponibles ---
rgb_dir = '../data/processed/rgb'
print("RGBs disponibles:")
for f in sorted(os.listdir(rgb_dir)):
    if f.endswith('.tif'):
        size = os.path.getsize(f'{rgb_dir}/{f}') / 1e6
        print(f"  {f}: {size:.1f} MB")

RGBs disponibles:
  CGSM_RGB_actual.tif: 121.5 MB
  CGSM_RGB_degradacion.tif: 123.2 MB
  CGSM_RGB_recuperacion.tif: 129.9 MB


In [34]:
# --- Inspeccionar RGB ---
import rasterio

rgb = '../data/processed/rgb/CGSM_RGB_degradacion.tif'
with rasterio.open(rgb) as src:
    print(f"Resolución: {src.res}")
    print(f"Tamaño: {src.width} x {src.height} px")
    print(f"CRS: {src.crs}")
    print(f"Bandas: {src.count}")
    print(f"Dtype: {src.dtypes[0]}")
    print(f"Tamaño en memoria estimado: {src.width * src.height * src.count * 4 / 1e9:.1f} GB")

Resolución: (8.983152841195215e-05, 8.983152841195215e-05)
Tamaño: 8684 x 8127 px
CRS: EPSG:4326
Bandas: 3
Dtype: uint8
Tamaño en memoria estimado: 0.8 GB


In [3]:
# --- Resample RGBs de 10m a 30m para SamGeo ---
import os
import rasterio
from rasterio.enums import Resampling

rgb_dir = '/home/rstudio/work/proyecto-cgsm/data/processed/rgb'
out_dir = '/home/rstudio/work/proyecto-cgsm/data/processed/rgb/resampled'
os.makedirs(out_dir, exist_ok=True)

scale_factor = 3  # 10m -> 30m

for nombre in ['degradacion', 'recuperacion', 'actual']:
    src_path = f'{rgb_dir}/CGSM_RGB_{nombre}.tif'
    dst_path = f'{out_dir}/CGSM_RGB_{nombre}_30m.tif'
    
    with rasterio.open(src_path) as src:
        new_h = src.height // scale_factor
        new_w = src.width // scale_factor
        
        data = src.read(
            out_shape=(src.count, new_h, new_w),
            resampling=Resampling.bilinear
        )
        
        transform = src.transform * src.transform.scale(
            src.width / new_w,
            src.height / new_h
        )
        
        profile = src.profile.copy()
        profile.update(width=new_w, height=new_h, transform=transform)
        
        with rasterio.open(dst_path, 'w', **profile) as dst:
            dst.write(data)
        
        size_mb = os.path.getsize(dst_path) / 1e6
        print(f"{nombre}: {src.width}x{src.height} -> {new_w}x{new_h} ({size_mb:.1f} MB)")

print("\nResample completado.")

degradacion: 8684x8127 -> 2894x2709 (13.9 MB)
recuperacion: 8684x8127 -> 2894x2709 (14.5 MB)
actual: 8684x8127 -> 2894x2709 (13.6 MB)

Resample completado.


In [ ]:
# --- Segmentación con SamGeo (vit_b, 30m) ---
from samgeo import SamGeo
import gc

res_dir = '/home/rstudio/work/proyecto-cgsm/data/processed/rgb/resampled'
out_dir = '/home/rstudio/work/proyecto-cgsm/data/processed/samgeo'
os.makedirs(out_dir, exist_ok=True)

print("Cargando SAM vit_b...")
sam = SamGeo(model_type="vit_b", automatic=True)
print("Modelo listo\n")

for nombre in ['degradacion', 'recuperacion', 'actual']:
    rgb_path = f'{res_dir}/CGSM_RGB_{nombre}_30m.tif'
    out_mask = f'{out_dir}/mask_{nombre}.tif'
    out_vector = f'{out_dir}/manglar_{nombre}.geojson'
    
    print(f"=== Segmentando {nombre} ===")
    try:
        sam.generate(rgb_path, output=out_mask)
        print(f"  Máscara OK")
        
        size_mb = os.path.getsize(out_mask) / 1e6
        print(f"  Tamaño: {size_mb:.1f} MB")
        
        sam.tiff_to_vector(out_mask, out_vector)
        print(f"  Vector OK\n")
    except Exception as e:
        print(f"  Error: {e}\n")

del sam
gc.collect()
print("=== Segmentación completada ===")

Cargando SAM vit_b...
Modelo listo

=== Segmentando degradacion ===


[ WARN:0@24.786] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 33550 (0x830e) encountered
[ WARN:0@24.786] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 33922 (0x8482) encountered
[ WARN:0@24.786] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34735 (0x87af) encountered
[ WARN:0@24.786] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34736 (0x87b0) encountered
[ WARN:0@24.786] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34737 (0x87b1) encountered


In [1]:
# --- Contar parches por período ---
import geopandas as gpd
out_dir = '/home/rstudio/work/proyecto-cgsm/data/processed/samgeo'
for nombre in ['degradacion', 'recuperacion', 'actual']:
    gdf = gpd.read_file(f'{out_dir}/manglar_{nombre}.geojson')
    print(f"{nombre}: {len(gdf)} parches")

degradacion: 1548 parches
recuperacion: 1170 parches
actual: 488 parches


In [3]:
# --- Clasificación espectral de parches ---
# Usar reduceRegions en batch para calcular NDVI y CMRI medio por parche
import ee
import geopandas as gpd
import math

ee.Initialize()

# Colección S2
def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    cmri = ndvi.subtract(ndwi).rename('CMRI')
    return image.addBands([ndvi, ndwi, cmri])

aoi = ee.Geometry.Polygon([
    [-74.8700, 11.0600], [-74.8200, 11.0700], [-74.7500, 11.0650],
    [-74.6800, 11.0600], [-74.6000, 11.0500], [-74.5300, 11.0400],
    [-74.4800, 11.0350], [-74.4300, 11.0300], [-74.3800, 11.0200],
    [-74.3200, 11.0100], [-74.2500, 10.9900], [-74.2000, 10.9500],
    [-74.1700, 10.9000], [-74.1500, 10.8500], [-74.1300, 10.8000],
    [-74.1200, 10.7500], [-74.1100, 10.7000], [-74.1000, 10.6500],
    [-74.1200, 10.6000], [-74.1500, 10.5500], [-74.2000, 10.5000],
    [-74.2500, 10.4500], [-74.3000, 10.4000], [-74.3500, 10.3500],
    [-74.4500, 10.3400], [-74.5500, 10.3800], [-74.6200, 10.4200],
    [-74.7000, 10.4800], [-74.7500, 10.5500], [-74.8000, 10.6500],
    [-74.8500, 10.7500], [-74.8700, 10.8500], [-74.8800, 10.9500],
    [-74.8700, 11.0600]
])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2_clouds)
      .map(add_indices))

composites = {
    'degradacion': s2.filterDate('2020-07-01', '2020-12-31').median().clip(aoi),
    'recuperacion': s2.filterDate('2022-01-01', '2022-06-30').median().clip(aoi),
    'actual': s2.filterDate('2024-07-01', '2025-06-30').median().clip(aoi),
}

out_dir = '/home/rstudio/work/proyecto-cgsm/data/processed/samgeo'
resultados = {}

for nombre in ['degradacion', 'recuperacion', 'actual']:
    print(f"\n=== Procesando {nombre} ===")
    gdf = gpd.read_file(f'{out_dir}/manglar_{nombre}.geojson')
    print(f"  Parches totales: {len(gdf)}")
    
    composite = composites[nombre]
    features = gdf.__geo_interface__['features']
    batch_size = 200
    all_results = []
    
    for i in range(0, len(features), batch_size):
        batch = features[i:i+batch_size]
        fc_batch = ee.FeatureCollection(batch)
        res = composite.select(['NDVI', 'CMRI']).reduceRegions(
            collection=fc_batch,
            reducer=ee.Reducer.mean(),
            scale=30
        ).getInfo()
        all_results.extend(res['features'])
        print(f"  Batch {i//batch_size + 1}/{math.ceil(len(features)/batch_size)} completado")
    
    gdf['NDVI_mean'] = [f['properties'].get('NDVI') for f in all_results]
    gdf['CMRI_mean'] = [f['properties'].get('CMRI') for f in all_results]
    gdf['area_ha'] = gdf.to_crs(epsg=32618).geometry.area / 10000
    
    # Filtrar parches válidos
    gdf_valid = gdf[
        (gdf['area_ha'] >= 1) &
        (gdf['area_ha'] < 5000) &
        (gdf['NDVI_mean'].notna())
    ].copy()
    gdf_valid['es_manglar'] = gdf_valid['CMRI_mean'] > 0
    
    manglar = gdf_valid[gdf_valid['es_manglar']]
    print(f"  Parches válidos (1-5000 ha): {len(gdf_valid)}")
    print(f"  Manglar: {len(manglar)} parches, {manglar['area_ha'].sum():.1f} ha")
    print(f"    NDVI medio: {manglar['NDVI_mean'].mean():.3f}")
    print(f"    CMRI medio: {manglar['CMRI_mean'].mean():.3f}")
    
    resultados[nombre] = gdf_valid

print("\n=== COMPARACIÓN MULTITEMPORAL ===")
print(f"{'Período':<20} {'Parches':>10} {'Área (ha)':>15} {'NDVI':>10} {'CMRI':>10}")
print("-" * 67)
for nombre, gdf in resultados.items():
    m = gdf[gdf['es_manglar']]
    print(f"{nombre:<20} {len(m):>10} {m['area_ha'].sum():>15,.1f} {m['NDVI_mean'].mean():>10.3f} {m['CMRI_mean'].mean():>10.3f}")


=== Procesando degradacion ===
  Parches totales: 1548
  Batch 1/8 completado
  Batch 2/8 completado
  Batch 3/8 completado
  Batch 4/8 completado
  Batch 5/8 completado
  Batch 6/8 completado
  Batch 7/8 completado
  Batch 8/8 completado
  Parches válidos (1-5000 ha): 29
  Manglar: 16 parches, 4938.5 ha
    NDVI medio: 0.503
    CMRI medio: 0.954

=== Procesando recuperacion ===
  Parches totales: 1170
  Batch 1/6 completado
  Batch 2/6 completado
  Batch 3/6 completado
  Batch 4/6 completado
  Batch 5/6 completado
  Batch 6/6 completado
  Parches válidos (1-5000 ha): 29
  Manglar: 16 parches, 5067.9 ha
    NDVI medio: 0.444
    CMRI medio: 0.882

=== Procesando actual ===
  Parches totales: 488
  Batch 1/3 completado
  Batch 2/3 completado
  Batch 3/3 completado
  Parches válidos (1-5000 ha): 49
  Manglar: 34 parches, 8444.5 ha
    NDVI medio: 0.564
    CMRI medio: 1.081

=== COMPARACIÓN MULTITEMPORAL ===
Período                 Parches       Área (ha)       NDVI       CMRI
--------

In [5]:
# --- Tabla comparativa multitemporal ---
print("=== COMPARACIÓN MULTITEMPORAL ===\n")
print(f"{'Período':<20} {'Parches':>10} {'Área manglar (ha)':>20} {'NDVI medio':>12} {'CMRI medio':>12}")
print("-" * 76)

for nombre, gdf in resultados.items():
    manglar = gdf[gdf['es_manglar']]
    print(f"{nombre:<20} {len(manglar):>10} {manglar['area_ha'].sum():>20,.1f} {manglar['NDVI_mean'].mean():>12.3f} {manglar['CMRI_mean'].mean():>12.3f}")

print("\n*** FASE 3 COMPLETA — Guardar notebook con Ctrl+S ***")

=== COMPARACIÓN MULTITEMPORAL ===

Período                 Parches    Área manglar (ha)   NDVI medio   CMRI medio
----------------------------------------------------------------------------
degradacion                  16              4,938.5        0.503        0.954
recuperacion                 16              5,067.9        0.444        0.882
actual                       34              8,444.5        0.564        1.081

*** FASE 3 COMPLETA — Guardar notebook con Ctrl+S ***


In [6]:
import pandas as pd

# Tabla comparativa
comparacion = pd.DataFrame({
    'Período': ['Degradación (2020-S2)', 'Recuperación (2022-S1)', 'Actual (2024-2025)'],
    'Parches': [16, 16, 34],
    'Área manglar (ha)': [4938.5, 5067.9, 8444.5],
    'NDVI medio': [0.503, 0.444, 0.564],
    'CMRI medio': [0.954, 0.882, 1.081]
})
comparacion.to_csv('../outputs/tables/comparacion_multitemporal.csv', index=False)
print("Tabla guardada en outputs/tables/comparacion_multitemporal.csv")
print(comparacion.to_string(index=False))

Tabla guardada en outputs/tables/comparacion_multitemporal.csv
               Período  Parches  Área manglar (ha)  NDVI medio  CMRI medio
 Degradación (2020-S2)       16             4938.5       0.503       0.954
Recuperación (2022-S1)       16             5067.9       0.444       0.882
    Actual (2024-2025)       34             8444.5       0.564       1.081


In [3]:
import ee
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import os

ee.Initialize()

aoi_coords = [
    [-74.8700, 11.0600], [-74.8200, 11.0700], [-74.7500, 11.0650],
    [-74.6800, 11.0600], [-74.6000, 11.0500], [-74.5300, 11.0400],
    [-74.4800, 11.0350], [-74.4300, 11.0300], [-74.3800, 11.0200],
    [-74.3200, 11.0100], [-74.2500, 10.9900], [-74.2000, 10.9500],
    [-74.1700, 10.9000], [-74.1500, 10.8500], [-74.1300, 10.8000],
    [-74.1200, 10.7500], [-74.1100, 10.7000], [-74.1000, 10.6500],
    [-74.1200, 10.6000], [-74.1500, 10.5500], [-74.2000, 10.5000],
    [-74.2500, 10.4500], [-74.3000, 10.4000], [-74.3500, 10.3500],
    [-74.4500, 10.3400], [-74.5500, 10.3800], [-74.6200, 10.4200],
    [-74.7000, 10.4800], [-74.7500, 10.5500], [-74.8000, 10.6500],
    [-74.8500, 10.7500], [-74.8700, 10.8500], [-74.8800, 10.9500],
    [-74.8700, 11.0600]
]
aoi = ee.Geometry.Polygon(aoi_coords)

# 1. Cargar GMW
print("=== Cargando Global Mangrove Watch ===")

# Intentar GMW v4.0 
try:
    gmw_2020 = ee.ImageCollection('projects/sat-io/open-datasets/GMW/annual-extent/GMW_MNG_2020').median().clip(aoi)
    gmw_area_raw = gmw_2020.gt(0).multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
    ).getInfo()
    gmw_area_km2 = list(gmw_area_raw.values())[0] / 1e6
    print(f"GMW v4.0 (2020) cargado")
    print(f"Área manglar GMW: {gmw_area_km2:.1f} km²")
    gmw_binary = gmw_2020.gt(0).rename('gmw')
except Exception as e1:
    print(f"GMW v4.0 falló: {e1}")
    try:
        gmw_2020 = ee.ImageCollection('projects/sat-io/open-datasets/GMW/extent/GMW_V3').median().clip(aoi)
        gmw_area_raw = gmw_2020.gt(0).multiply(ee.Image.pixelArea()).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
        ).getInfo()
        gmw_area_km2 = list(gmw_area_raw.values())[0] / 1e6
        print(f"GMW v3.0 cargado")
        print(f"Área manglar GMW: {gmw_area_km2:.1f} km²")
        gmw_binary = gmw_2020.gt(0).rename('gmw')
    except Exception as e2:
        print(f"GMW v3.0 falló: {e2}")
        gmw_2020 = ee.ImageCollection('LANDSAT/MANGROVE_FORESTS').first().clip(aoi)
        gmw_area_raw = gmw_2020.select('1').multiply(ee.Image.pixelArea()).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
        ).getInfo()
        gmw_area_km2 = list(gmw_area_raw.values())[0] / 1e6
        print(f"Landsat Mangrove Forests (Giri 2011) cargado como alternativa")
        print(f"Área manglar: {gmw_area_km2:.1f} km²")
        gmw_binary = gmw_2020.select('1').gt(0).rename('gmw')

# 2. clasificación 2020
print("\n=== Generando clasificación 2020 ===")

def mask_s2(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    cmri = ndvi.subtract(ndwi).rename('CMRI')
    return image.addBands([ndvi, ndwi, cmri])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2)
      .map(add_indices))

comp_2020 = s2.filterDate('2020-01-01', '2020-12-31').median().clip(aoi)
ndvi_2020 = comp_2020.normalizedDifference(['B8', 'B4'])
ndwi_2020 = comp_2020.normalizedDifference(['B3', 'B8'])
cmri_2020 = ndvi_2020.subtract(ndwi_2020)
our_manglar = cmri_2020.gt(0.3).And(ndvi_2020.gt(0.4)).rename('manglar')

our_area_val = our_manglar.selfMask().multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).get('manglar').getInfo()
our_area_km2 = our_area_val / 1e6
print(f"Área manglar nuestra clasificación 2020: {our_area_km2:.1f} km²")

# 3. IoU pixel a pixel
print("\n=== Calculando IoU ===")
our_binary = our_manglar.unmask(0).rename('ours')

intersection = gmw_binary.And(our_binary).rename('inter')
inter_area = intersection.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).get('inter').getInfo()

union_img = gmw_binary.Or(our_binary).rename('union')
union_area = union_img.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).get('union').getInfo()

iou = inter_area / union_area if union_area > 0 else 0
print(f"Intersección: {inter_area/1e6:.1f} km²")
print(f"Unión: {union_area/1e6:.1f} km²")
print(f"IoU: {iou:.3f}")

# 4. Matriz de confusión por muestreo
print("\n=== Matriz de confusión (muestreo aleatorio) ===")
combined = gmw_binary.addBands(our_binary)

sample_points = combined.stratifiedSample(
    numPoints=500,
    classBand='gmw',
    region=aoi,
    scale=30,
    seed=42,
    geometries=True
).getInfo()

y_true = []
y_pred = []
for feat in sample_points['features']:
    props = feat['properties']
    y_true.append(int(props.get('gmw', 0)))
    y_pred.append(int(props.get('ours', 0)))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
print(f"\nMatriz de confusión (GMW vs Nuestra clasificación):")
print(f"                    Pred: No manglar  Pred: Manglar")
print(f"  Real: No manglar      {cm[0,0]:>8}       {cm[0,1]:>8}")
print(f"  Real: Manglar         {cm[1,0]:>8}       {cm[1,1]:>8}")

tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)

print(f"\n=== MÉTRICAS DE VALIDACIÓN ===")
print(f"  Precisión global (OA): {accuracy:.3f}")
print(f"  Precisión (Producer's): {precision:.3f}")
print(f"  Recall (User's): {recall:.3f}")
print(f"  F1-score: {f1:.3f}")
print(f"  IoU: {iou:.3f}")

# 5. Guardar
os.makedirs('../outputs/tables', exist_ok=True)
metrics_df = pd.DataFrame({
    'Métrica': [
        'Precisión global (OA)',
        "Precisión (Producer's)",
        "Recall (User's)",
        'F1-score',
        'IoU',
        'Área GMW (km²)',
        'Área nuestra (km²)',
        'Intersección (km²)',
        'Unión (km²)',
        'TP', 'TN', 'FP', 'FN'
    ],
    'Valor': [
        round(accuracy, 3),
        round(precision, 3),
        round(recall, 3),
        round(f1, 3),
        round(iou, 3),
        round(gmw_area_km2, 1),
        round(our_area_km2, 1),
        round(inter_area/1e6, 1),
        round(union_area/1e6, 1),
        tp, tn, fp, fn
    ]
})
metrics_df.to_csv('../outputs/tables/validacion_gmw.csv', index=False)
print(f"\nResultados guardados en outputs/tables/validacion_gmw.csv")

=== Cargando Global Mangrove Watch ===
GMW v4.0 (2020) cargado
Área manglar GMW: 376.5 km²

=== Generando clasificación 2020 ===
Área manglar nuestra clasificación 2020: 3032.9 km²

=== Calculando IoU ===
Intersección: 356.4 km²
Unión: 376.5 km²
IoU: 0.947

=== Matriz de confusión (muestreo aleatorio) ===

Matriz de confusión (GMW vs Nuestra clasificación):
                    Pred: No manglar  Pred: Manglar
  Real: No manglar             0              0
  Real: Manglar               42            458

=== MÉTRICAS DE VALIDACIÓN ===
  Precisión global (OA): 0.916
  Precisión (Producer's): 1.000
  Recall (User's): 0.916
  F1-score: 0.956
  IoU: 0.947

Resultados guardados en outputs/tables/validacion_gmw.csv


In [4]:
# --- Muestreo corregido: forzar ambas clases ---
print("=== Muestreo corregido ===")

# Crear imagen binaria donde 0 = no manglar, 1 = manglar (sin nodata)
gmw_full = gmw_binary.unmask(0)
our_full = our_binary.unmask(0)
combined = gmw_full.addBands(our_full)

# Muestreo aleatorio simple (no estratificado)
sample = combined.sample(
    region=aoi,
    scale=30,
    numPixels=2000,
    seed=42,
    geometries=False
).getInfo()

y_true = []
y_pred = []
for feat in sample['features']:
    props = feat['properties']
    y_true.append(int(props.get('gmw', 0)))
    y_pred.append(int(props.get('ours', 0)))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
print(f"\nMatriz de confusión corregida:")
print(f"                    Pred: No manglar  Pred: Manglar")
print(f"  Real: No manglar      {cm[0,0]:>8}       {cm[0,1]:>8}")
print(f"  Real: Manglar         {cm[1,0]:>8}       {cm[1,1]:>8}")

tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)

print(f"\n=== MÉTRICAS CORREGIDAS ===")
print(f"  Precisión global (OA): {accuracy:.3f}")
print(f"  Precisión (Producer's): {precision:.3f}")
print(f"  Recall (User's): {recall:.3f}")
print(f"  F1-score: {f1:.3f}")
print(f"  IoU: {iou:.3f}")
print(f"  Puntos muestreados: {len(y_true)}")
print(f"  No manglar: {(y_true==0).sum()} | Manglar: {(y_true==1).sum()}")

# Actualizar CSV
metrics_df = pd.DataFrame({
    'Métrica': [
        'Precisión global (OA)', "Precisión (Producer's)", "Recall (User's)",
        'F1-score', 'IoU', 'Área GMW (km²)', 'Área nuestra (km²)',
        'Intersección (km²)', 'Unión (km²)', 'TP', 'TN', 'FP', 'FN',
        'Total puntos muestreados'
    ],
    'Valor': [
        round(accuracy, 3), round(precision, 3), round(recall, 3),
        round(f1, 3), round(iou, 3), round(gmw_area_km2, 1), round(our_area_km2, 1),
        round(inter_area/1e6, 1), round(union_area/1e6, 1),
        tp, tn, fp, fn, len(y_true)
    ]
})
metrics_df.to_csv('../outputs/tables/validacion_gmw.csv', index=False)
print("\nCSV actualizado")

=== Muestreo corregido ===

Matriz de confusión corregida:
                    Pred: No manglar  Pred: Manglar
  Real: No manglar           735           1090
  Real: Manglar               14            161

=== MÉTRICAS CORREGIDAS ===
  Precisión global (OA): 0.448
  Precisión (Producer's): 0.129
  Recall (User's): 0.920
  F1-score: 0.226
  IoU: 0.947
  Puntos muestreados: 2000
  No manglar: 1825 | Manglar: 175

CSV actualizado


In [2]:
import ee
import numpy as np
from sklearn.metrics import confusion_matrix
ee.Initialize()

aoi_coords = [
    [-74.8700, 11.0600], [-74.8200, 11.0700], [-74.7500, 11.0650],
    [-74.6800, 11.0600], [-74.6000, 11.0500], [-74.5300, 11.0400],
    [-74.4800, 11.0350], [-74.4300, 11.0300], [-74.3800, 11.0200],
    [-74.3200, 11.0100], [-74.2500, 10.9900], [-74.2000, 10.9500],
    [-74.1700, 10.9000], [-74.1500, 10.8500], [-74.1300, 10.8000],
    [-74.1200, 10.7500], [-74.1100, 10.7000], [-74.1000, 10.6500],
    [-74.1200, 10.6000], [-74.1500, 10.5500], [-74.2000, 10.5000],
    [-74.2500, 10.4500], [-74.3000, 10.4000], [-74.3500, 10.3500],
    [-74.4500, 10.3400], [-74.5500, 10.3800], [-74.6200, 10.4200],
    [-74.7000, 10.4800], [-74.7500, 10.5500], [-74.8000, 10.6500],
    [-74.8500, 10.7500], [-74.8700, 10.8500], [-74.8800, 10.9500],
    [-74.8700, 11.0600]
]
aoi = ee.Geometry.Polygon(aoi_coords)

def mask_s2(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    cmri = ndvi.subtract(ndwi).rename('CMRI')
    return image.addBands([ndvi, ndwi, cmri])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2).map(add_indices))

comp_2020 = s2.filterDate('2020-01-01', '2020-12-31').median().clip(aoi)
ndvi_2020 = comp_2020.normalizedDifference(['B8', 'B4'])
ndwi_2020 = comp_2020.normalizedDifference(['B3', 'B8'])
cmri_2020 = ndvi_2020.subtract(ndwi_2020)

gmw_2020 = ee.ImageCollection('projects/sat-io/open-datasets/GMW/annual-extent/GMW_MNG_2020').median().clip(aoi)
gmw_full = gmw_2020.gt(0).unmask(0).rename('gmw')

In [3]:
# --- Optimización de umbrales ---
print("=== Prueba de umbrales ===\n")
print(f"{'CMRI':>6} {'NDVI':>6} {'Área (km²)':>12} {'OA':>8} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 62)

best_f1 = 0
best_params = None

for cmri_th in [0.3, 0.5, 0.7, 0.9, 1.0]:
    for ndvi_th in [0.4, 0.5, 0.6, 0.7]:
        test_manglar = cmri_2020.gt(cmri_th).And(ndvi_2020.gt(ndvi_th)).rename('test')
        test_binary = test_manglar.unmask(0)
        
        test_combined = gmw_full.addBands(test_binary.rename('ours'))
        test_sample = test_combined.sample(
            region=aoi, scale=30, numPixels=1000, seed=42, geometries=False
        ).getInfo()
        
        yt = [int(f['properties'].get('gmw', 0)) for f in test_sample['features']]
        yp = [int(f['properties'].get('ours', 0)) for f in test_sample['features']]
        yt = np.array(yt)
        yp = np.array(yp)
        
        if len(np.unique(yt)) < 2 or len(np.unique(yp)) < 2:
            continue
        
        tcm = confusion_matrix(yt, yp)
        ttn, tfp, tfn, ttp = tcm.ravel()
        tp_ = ttp / (ttp + tfp) if (ttp + tfp) > 0 else 0
        tr_ = ttp / (ttp + tfn) if (ttp + tfn) > 0 else 0
        tf1 = 2 * tp_ * tr_ / (tp_ + tr_) if (tp_ + tr_) > 0 else 0
        toa = (ttp + ttn) / (ttp + ttn + tfp + tfn)
        
        # Área
        t_area = test_manglar.selfMask().multiply(ee.Image.pixelArea()).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=aoi, scale=100, maxPixels=1e13
        ).get('test').getInfo()
        t_area_km = t_area / 1e6 if t_area else 0
        
        print(f"{cmri_th:>6.1f} {ndvi_th:>6.1f} {t_area_km:>12.1f} {toa:>8.3f} {tp_:>10.3f} {tr_:>8.3f} {tf1:>8.3f}")
        
        if tf1 > best_f1:
            best_f1 = tf1
            best_params = (cmri_th, ndvi_th, t_area_km, toa, tp_, tr_, tf1)

print(f"\n=== MEJOR COMBINACIÓN ===")
print(f"  CMRI > {best_params[0]}, NDVI > {best_params[1]}")
print(f"  Área: {best_params[2]:.1f} km²")
print(f"  OA: {best_params[3]:.3f} | Precision: {best_params[4]:.3f} | Recall: {best_params[5]:.3f} | F1: {best_params[6]:.3f}")

=== Prueba de umbrales ===

  CMRI   NDVI   Área (km²)       OA  Precision   Recall       F1
--------------------------------------------------------------
   0.3    0.4       3103.6    0.454      0.122    0.915    0.216
   0.3    0.5       2382.9    0.562      0.138    0.829    0.237
   0.3    0.6       1722.5    0.682      0.172    0.756    0.281
   0.3    0.7       1109.0    0.785      0.217    0.622    0.322
   0.5    0.4       3103.6    0.454      0.122    0.915    0.216
   0.5    0.5       2382.9    0.562      0.138    0.829    0.237
   0.5    0.6       1722.5    0.682      0.172    0.756    0.281
   0.5    0.7       1109.0    0.785      0.217    0.622    0.322
   0.7    0.4       3100.5    0.454      0.122    0.915    0.216
   0.7    0.5       2382.9    0.562      0.138    0.829    0.237
   0.7    0.6       1722.5    0.682      0.172    0.756    0.281
   0.7    0.7       1109.0    0.785      0.217    0.622    0.322
   0.9    0.4       2970.6    0.464      0.117    0.841    0.205

In [5]:
import ee
import numpy as np
from sklearn.metrics import confusion_matrix
ee.Initialize()

aoi_coords = [
    [-74.8700, 11.0600], [-74.8200, 11.0700], [-74.7500, 11.0650],
    [-74.6800, 11.0600], [-74.6000, 11.0500], [-74.5300, 11.0400],
    [-74.4800, 11.0350], [-74.4300, 11.0300], [-74.3800, 11.0200],
    [-74.3200, 11.0100], [-74.2500, 10.9900], [-74.2000, 10.9500],
    [-74.1700, 10.9000], [-74.1500, 10.8500], [-74.1300, 10.8000],
    [-74.1200, 10.7500], [-74.1100, 10.7000], [-74.1000, 10.6500],
    [-74.1200, 10.6000], [-74.1500, 10.5500], [-74.2000, 10.5000],
    [-74.2500, 10.4500], [-74.3000, 10.4000], [-74.3500, 10.3500],
    [-74.4500, 10.3400], [-74.5500, 10.3800], [-74.6200, 10.4200],
    [-74.7000, 10.4800], [-74.7500, 10.5500], [-74.8000, 10.6500],
    [-74.8500, 10.7500], [-74.8700, 10.8500], [-74.8800, 10.9500],
    [-74.8700, 11.0600]
]
aoi = ee.Geometry.Polygon(aoi_coords)

def mask_s2(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    cmri = ndvi.subtract(ndwi).rename('CMRI')
    return image.addBands([ndvi, ndwi, cmri])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2).map(add_indices))

comp_2020 = s2.filterDate('2020-01-01', '2020-12-31').median().clip(aoi)
ndvi_2020 = comp_2020.normalizedDifference(['B8', 'B4'])
ndwi_2020 = comp_2020.normalizedDifference(['B3', 'B8'])
cmri_2020 = ndvi_2020.subtract(ndwi_2020)

gmw_2020 = ee.ImageCollection('projects/sat-io/open-datasets/GMW/annual-extent/GMW_MNG_2020').median().clip(aoi)
gmw_full = gmw_2020.gt(0).unmask(0).rename('gmw')

# Restricciones espaciales
srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi)
elev_mask = srtm.lt(10)

jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').clip(aoi)
water = jrc.gt(30)
water_dist = water.fastDistanceTransform().sqrt().multiply(30)
near_water = water_dist.lt(3000)

print("=== Clasificación mejorada con restricciones espaciales ===\n")
print(f"{'Config':<40} {'Área km²':>10} {'OA':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
print("-" * 82)

configs = [
    ('CMRI>0.3 NDVI>0.7 (baseline)',
     cmri_2020.gt(0.3).And(ndvi_2020.gt(0.7))),
    ('+ elev<10m',
     cmri_2020.gt(0.3).And(ndvi_2020.gt(0.7)).And(elev_mask)),
    ('+ elev<10m + agua<3km',
     cmri_2020.gt(0.3).And(ndvi_2020.gt(0.7)).And(elev_mask).And(near_water)),
    ('CMRI>0.5 NDVI>0.6 + elev + agua',
     cmri_2020.gt(0.5).And(ndvi_2020.gt(0.6)).And(elev_mask).And(near_water)),
    ('CMRI>0.3 NDVI>0.6 + elev + agua',
     cmri_2020.gt(0.3).And(ndvi_2020.gt(0.6)).And(elev_mask).And(near_water)),
    ('CMRI>0.3 NDVI>0.5 + elev + agua',
     cmri_2020.gt(0.3).And(ndvi_2020.gt(0.5)).And(elev_mask).And(near_water)),
    ('CMRI>0.7 NDVI>0.5 + elev + agua',
     cmri_2020.gt(0.7).And(ndvi_2020.gt(0.5)).And(elev_mask).And(near_water)),
    ('CMRI>1.0 NDVI>0.5 + elev + agua',
     cmri_2020.gt(1.0).And(ndvi_2020.gt(0.5)).And(elev_mask).And(near_water)),
    ('CMRI>0.5 NDVI>0.5 + elev + agua',
     cmri_2020.gt(0.5).And(ndvi_2020.gt(0.5)).And(elev_mask).And(near_water)),
    ('CMRI>0.3 NDVI>0.4 + elev + agua',
     cmri_2020.gt(0.3).And(ndvi_2020.gt(0.4)).And(elev_mask).And(near_water)),
]

best_f1 = 0
best_name = ''
best_metrics = {}

for name, mask in configs:
    try:
        test = mask.rename('test').unmask(0)
        combined_test = gmw_full.addBands(test.rename('ours'))
        
        samp = combined_test.sample(
            region=aoi, scale=30, numPixels=1500, seed=42, geometries=False
        ).getInfo()
        
        yt = np.array([int(f['properties'].get('gmw', 0)) for f in samp['features']])
        yp = np.array([int(f['properties'].get('ours', 0)) for f in samp['features']])
        
        if len(np.unique(yt)) < 2 or len(np.unique(yp)) < 2:
            print(f"{name:<40} {'skip':>10} {'---':>8} {'---':>8} {'---':>8} {'---':>8}")
            continue
        
        tcm = confusion_matrix(yt, yp)
        ttn, tfp, tfn, ttp = tcm.ravel()
        prec = ttp / (ttp + tfp) if (ttp + tfp) > 0 else 0
        rec = ttp / (ttp + tfn) if (ttp + tfn) > 0 else 0
        tf1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        toa = (ttp + ttn) / (ttp + ttn + tfp + tfn)
        
        # Área con protección
        try:
            area_dict = mask.selfMask().rename('area').multiply(ee.Image.pixelArea()).reduceRegion(
                reducer=ee.Reducer.sum(), geometry=aoi, scale=100, maxPixels=1e13
            ).getInfo()
            t_area_km = area_dict.get('area', 0) / 1e6 if area_dict.get('area') else 0
        except:
            t_area_km = 0
        
        print(f"{name:<40} {t_area_km:>10.1f} {toa:>8.3f} {prec:>8.3f} {rec:>8.3f} {tf1:>8.3f}")
        
        if tf1 > best_f1:
            best_f1 = tf1
            best_name = name
            best_metrics = {
                'area': t_area_km, 'oa': toa, 'prec': prec,
                'rec': rec, 'f1': tf1, 'tp': ttp, 'tn': ttn, 'fp': tfp, 'fn': tfn
            }
    except Exception as e:
        print(f"{name:<40} ERROR: {str(e)[:40]}")

print(f"\n=== MEJOR CONFIGURACIÓN ===")
print(f"  {best_name}")
print(f"  Área: {best_metrics['area']:.1f} km²")
print(f"  OA: {best_metrics['oa']:.3f} | Precision: {best_metrics['prec']:.3f} | Recall: {best_metrics['rec']:.3f} | F1: {best_metrics['f1']:.3f}")
print(f"  TP: {best_metrics['tp']} | TN: {best_metrics['tn']} | FP: {best_metrics['fp']} | FN: {best_metrics['fn']}")

=== Clasificación mejorada con restricciones espaciales ===

Config                                     Área km²       OA     Prec      Rec       F1
----------------------------------------------------------------------------------
CMRI>0.3 NDVI>0.7 (baseline)                 1109.0    0.781    0.214    0.591    0.314
+ elev<10m                                    570.1    0.865    0.297    0.433    0.353
+ elev<10m + agua<3km                         556.0    0.896    0.396    0.433    0.414
CMRI>0.5 NDVI>0.6 + elev + agua               976.4    0.848    0.296    0.575    0.390
CMRI>0.3 NDVI>0.6 + elev + agua               976.4    0.848    0.296    0.575    0.390
CMRI>0.3 NDVI>0.5 + elev + agua              1334.5    0.803    0.248    0.654    0.359
CMRI>0.7 NDVI>0.5 + elev + agua              1334.4    0.803    0.248    0.654    0.359
CMRI>1.0 NDVI>0.5 + elev + agua              1300.1    0.801    0.236    0.606    0.340
CMRI>0.5 NDVI>0.5 + elev + agua              1334.5    0.803    

In [6]:
import pandas as pd

final = pd.DataFrame({
    'Métrica': [
        'Precisión global (OA)', "Precisión (Producer's)", "Recall (User's)", 'F1-score',
        'Área GMW 2020 (km²)', 'Área clasificación (km²)', 'Ratio área',
        'TP', 'TN', 'FP', 'FN',
        'Umbral CMRI', 'Umbral NDVI', 'Restricción elevación', 'Restricción distancia agua'
    ],
    'Valor': [
        0.896, 0.396, 0.433, 0.414,
        376.5, 556.0, round(556.0/376.5, 2),
        55, 1289, 84, 72,
        '> 0.3', '> 0.7', '< 10 m (SRTM)', '< 3 km (JRC)'
    ]
})
final.to_csv('../outputs/tables/validacion_gmw_final.csv', index=False)
print("Métricas finales guardadas")
print(final.to_string(index=False))

Métricas finales guardadas
                   Métrica         Valor
     Precisión global (OA)         0.896
    Precisión (Producer's)         0.396
           Recall (User's)         0.433
                  F1-score         0.414
       Área GMW 2020 (km²)         376.5
  Área clasificación (km²)         556.0
                Ratio área          1.48
                        TP            55
                        TN          1289
                        FP            84
                        FN            72
               Umbral CMRI         > 0.3
               Umbral NDVI         > 0.7
     Restricción elevación < 10 m (SRTM)
Restricción distancia agua  < 3 km (JRC)


In [7]:
# --- Refinamiento fino alrededor de la mejor configuración ---
print("=== Refinamiento fino: NDVI + CMRI + elev<10m + agua<3km ===\n")
print(f"{'CMRI':>6} {'NDVI':>6} {'Área km²':>10} {'OA':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
print("-" * 58)

best_f1 = 0
best_params = None

for cmri_th in [0.2, 0.3, 0.4, 0.5, 0.6]:
    for ndvi_th in [0.60, 0.65, 0.70, 0.75, 0.80]:
        try:
            mask = cmri_2020.gt(cmri_th).And(ndvi_2020.gt(ndvi_th)).And(elev_mask).And(near_water)
            test = mask.rename('test').unmask(0)
            combined_test = gmw_full.addBands(test.rename('ours'))
            
            samp = combined_test.sample(
                region=aoi, scale=30, numPixels=2000, seed=42, geometries=False
            ).getInfo()
            
            yt = np.array([int(f['properties'].get('gmw', 0)) for f in samp['features']])
            yp = np.array([int(f['properties'].get('ours', 0)) for f in samp['features']])
            
            if len(np.unique(yt)) < 2 or len(np.unique(yp)) < 2:
                continue
            
            tcm = confusion_matrix(yt, yp)
            ttn, tfp, tfn, ttp = tcm.ravel()
            prec = ttp / (ttp + tfp) if (ttp + tfp) > 0 else 0
            rec = ttp / (ttp + tfn) if (ttp + tfn) > 0 else 0
            tf1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
            toa = (ttp + ttn) / (ttp + ttn + tfp + tfn)
            
            try:
                area_dict = mask.selfMask().rename('area').multiply(ee.Image.pixelArea()).reduceRegion(
                    reducer=ee.Reducer.sum(), geometry=aoi, scale=100, maxPixels=1e13
                ).getInfo()
                t_area = area_dict.get('area', 0) / 1e6 if area_dict.get('area') else 0
            except:
                t_area = 0
            
            print(f"{cmri_th:>6.1f} {ndvi_th:>6.2f} {t_area:>10.1f} {toa:>8.3f} {prec:>8.3f} {rec:>8.3f} {tf1:>8.3f}")
            
            if tf1 > best_f1:
                best_f1 = tf1
                best_params = (cmri_th, ndvi_th, t_area, toa, prec, rec, tf1, ttp, ttn, tfp, tfn)
        except Exception as e:
            pass

print(f"\n=== MEJOR COMBINACIÓN REFINADA ===")
print(f"  CMRI > {best_params[0]}, NDVI > {best_params[1]}, elev<10m, agua<3km")
print(f"  Área: {best_params[2]:.1f} km²")
print(f"  OA: {best_params[3]:.3f} | Precision: {best_params[4]:.3f} | Recall: {best_params[5]:.3f} | F1: {best_params[6]:.3f}")
print(f"  TP: {best_params[7]} | TN: {best_params[8]} | FP: {best_params[9]} | FN: {best_params[10]}")

=== Refinamiento fino: NDVI + CMRI + elev<10m + agua<3km ===

  CMRI   NDVI   Área km²       OA     Prec      Rec       F1
----------------------------------------------------------
   0.2   0.60      976.4    0.855    0.321    0.589    0.415
   0.2   0.65      765.4    0.877    0.361    0.520    0.426
   0.2   0.70      556.0    0.899    0.428    0.457    0.442
   0.2   0.75      359.7    0.909    0.471    0.320    0.381
   0.2   0.80      173.1    0.907    0.414    0.137    0.206
   0.3   0.60      976.4    0.855    0.321    0.589    0.415
   0.3   0.65      765.4    0.877    0.361    0.520    0.426
   0.3   0.70      556.0    0.899    0.428    0.457    0.442
   0.3   0.75      359.7    0.909    0.471    0.320    0.381
   0.3   0.80      173.1    0.907    0.414    0.137    0.206
   0.4   0.60      976.4    0.855    0.321    0.589    0.415
   0.4   0.65      765.4    0.877    0.361    0.520    0.426
   0.4   0.70      556.0    0.899    0.428    0.457    0.442
   0.4   0.75      359.7 

In [8]:
import ee
import geemap
import os

ee.Initialize()

aoi_coords = [
    [-74.8700, 11.0600], [-74.8200, 11.0700], [-74.7500, 11.0650],
    [-74.6800, 11.0600], [-74.6000, 11.0500], [-74.5300, 11.0400],
    [-74.4800, 11.0350], [-74.4300, 11.0300], [-74.3800, 11.0200],
    [-74.3200, 11.0100], [-74.2500, 10.9900], [-74.2000, 10.9500],
    [-74.1700, 10.9000], [-74.1500, 10.8500], [-74.1300, 10.8000],
    [-74.1200, 10.7500], [-74.1100, 10.7000], [-74.1000, 10.6500],
    [-74.1200, 10.6000], [-74.1500, 10.5500], [-74.2000, 10.5000],
    [-74.2500, 10.4500], [-74.3000, 10.4000], [-74.3500, 10.3500],
    [-74.4500, 10.3400], [-74.5500, 10.3800], [-74.6200, 10.4200],
    [-74.7000, 10.4800], [-74.7500, 10.5500], [-74.8000, 10.6500],
    [-74.8500, 10.7500], [-74.8700, 10.8500], [-74.8800, 10.9500],
    [-74.8700, 11.0600]
]
aoi = ee.Geometry.Polygon(aoi_coords)

def mask_s2(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    cmri = ndvi.subtract(ndwi).rename('CMRI')
    return image.addBands([ndvi, ndwi, cmri])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2).map(add_indices))

# Restricciones espaciales
srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi)
elev_mask = srtm.lt(10)
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').clip(aoi)
water = jrc.gt(30)
water_dist = water.fastDistanceTransform().sqrt().multiply(30)
near_water = water_dist.lt(3000)

# --- Clasificación manglar por período con umbrales optimizados ---
def manglar_mask(start, end):
    comp = s2.filterDate(start, end).median().clip(aoi)
    ndvi = comp.normalizedDifference(['B8', 'B4'])
    return ndvi.gt(0.70).And(elev_mask).And(near_water).selfMask()

manglar_deg = manglar_mask('2020-07-01', '2020-12-31').rename('deg')
manglar_act = manglar_mask('2024-07-01', '2025-06-30').rename('act')

# --- Mapa de cambio: ganancia, pérdida, estable ---
# 0 = sin manglar en ningún período
# 1 = pérdida (manglar en degradación, no en actual) — ROJO
# 2 = estable (manglar en ambos) — VERDE
# 3 = ganancia (no manglar en degradación, sí en actual) — AZUL

deg_binary = manglar_deg.unmask(0).gt(0)
act_binary = manglar_act.unmask(0).gt(0)

perdida = deg_binary.And(act_binary.Not()).selfMask().multiply(1)
estable = deg_binary.And(act_binary).selfMask().multiply(2)
ganancia = deg_binary.Not().And(act_binary).selfMask().multiply(3)

cambio = perdida.unmask(0).add(estable.unmask(0)).add(ganancia.unmask(0)).selfMask().rename('cambio')

# Calcular áreas
print("=== MAPA DE GANANCIA/PÉRDIDA DE MANGLAR ===\n")
print("Degradación (2020-S2) vs Actual (2024-2025)\n")

area_deg = deg_binary.selfMask().multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).getInfo()

area_act = act_binary.selfMask().multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).getInfo()

area_perdida = perdida.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).getInfo()

area_estable = estable.gt(0).selfMask().multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).getInfo()

area_ganancia = ganancia.gt(0).selfMask().multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).getInfo()

deg_km2 = list(area_deg.values())[0] / 1e6
act_km2 = list(area_act.values())[0] / 1e6
per_km2 = list(area_perdida.values())[0] / 1e6
est_km2 = list(area_estable.values())[0] / 1e6
gan_km2 = list(area_ganancia.values())[0] / 1e6
neto_km2 = gan_km2 - per_km2

print(f"  Manglar degradación (2020): {deg_km2:.1f} km²")
print(f"  Manglar actual (2024-2025): {act_km2:.1f} km²")
print(f"")
print(f"  Pérdida (rojo):   {per_km2:.1f} km²")
print(f"  Estable (verde):  {est_km2:.1f} km²")
print(f"  Ganancia (azul):  {gan_km2:.1f} km²")
print(f"  Cambio neto:      {'+' if neto_km2 > 0 else ''}{neto_km2:.1f} km²")

# --- Visualización ---
from ipyleaflet import Marker, DivIcon

Map = geemap.Map(center=[10.72, -74.45], zoom=10, layout={'height': '700px'})
Map.add_basemap('Esri.WorldImagery')

# Capa de cambio
vis_cambio = {'min': 1, 'max': 3, 'palette': ['#D32F2F', '#2E7D32', '#1565C0']}
Map.addLayer(cambio, vis_cambio, 'Cambio manglar (rojo=pérdida, verde=estable, azul=ganancia)')

# Capas individuales
Map.addLayer(perdida, {'palette': ['#D32F2F']}, 'Pérdida', shown=False, opacity=0.8)
Map.addLayer(estable, {'palette': ['#2E7D32']}, 'Estable', shown=False, opacity=0.8)
Map.addLayer(ganancia, {'palette': ['#1565C0']}, 'Ganancia', shown=False, opacity=0.8)

# AOI
styled_aoi = ee.FeatureCollection([ee.Feature(aoi)]).style(color='FFFFFF', fillColor='00000000', width=2)
Map.addLayer(styled_aoi, {}, 'AOI')

# Leyenda HTML
from ipyleaflet import WidgetControl
from ipywidgets import HTML

legend_html = """
<div style="background:rgba(255,255,255,0.9); padding:10px 14px; border-radius:6px; border:1px solid #999; font-family:Arial; font-size:12px; line-height:24px;">
    <b>Cambio de manglar</b><br>
    <b>2020 (degradación) → 2024-2025 (actual)</b><br><br>
    <svg width="14" height="14" style="vertical-align:middle;"><rect width="14" height="14" fill="#D32F2F"/></svg> Pérdida (""" + f"{per_km2:.1f}" + """ km²)<br>
    <svg width="14" height="14" style="vertical-align:middle;"><rect width="14" height="14" fill="#2E7D32"/></svg> Estable (""" + f"{est_km2:.1f}" + """ km²)<br>
    <svg width="14" height="14" style="vertical-align:middle;"><rect width="14" height="14" fill="#1565C0"/></svg> Ganancia (""" + f"{gan_km2:.1f}" + """ km²)<br>
    <br><b>Cambio neto: """ + f"{'+' if neto_km2 > 0 else ''}{neto_km2:.1f}" + """ km²</b>
</div>
"""
Map.add(WidgetControl(widget=HTML(value=legend_html), position='bottomleft'))
Map.add_layer_control()

# Guardar
import pandas as pd
cambio_df = pd.DataFrame({
    'Categoría': ['Manglar degradación (2020)', 'Manglar actual (2024-2025)',
                  'Pérdida', 'Estable', 'Ganancia', 'Cambio neto'],
    'Área (km²)': [round(deg_km2, 1), round(act_km2, 1),
                   round(per_km2, 1), round(est_km2, 1), round(gan_km2, 1), round(neto_km2, 1)]
})
os.makedirs('../outputs/tables', exist_ok=True)
cambio_df.to_csv('../outputs/tables/ganancia_perdida_manglar.csv', index=False)

print(f"\nDatos guardados en outputs/tables/ganancia_perdida_manglar.csv")
print("Mapa listo")
Map

=== MAPA DE GANANCIA/PÉRDIDA DE MANGLAR ===

Degradación (2020-S2) vs Actual (2024-2025)

  Manglar degradación (2020): 874.1 km²
  Manglar actual (2024-2025): 950.1 km²

  Pérdida (rojo):   183.2 km²
  Estable (verde):  690.9 km²
  Ganancia (azul):  259.2 km²
  Cambio neto:      +76.0 km²

Datos guardados en outputs/tables/ganancia_perdida_manglar.csv
Mapa listo


Map(center=[10.72, -74.45], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…